# Trial-Based Synchronized Video Creation

Creates separate video files for each trial from `trials_data.csv` with synchronized traces:
- Left/right eye k_phi and k_theta traces
- Pupil diameter for left and right eyes
- Bug trajectory (x, y position)
- Full-resolution LFP trace (not downsampled)

Based on `synchronized_video_creation.ipynb` but adapted for trial-based export.

In [8]:
import datetime
import numpy as np
import cv2
from itertools import cycle
import pickle
import pathlib
import scipy.io
from matplotlib import pyplot as plt
import pandas as pd
from pathlib import Path
from typing import Optional, Sequence, Dict, Tuple, Union, Literal
from tqdm import tqdm

from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from eye_tracking_system_tools.preprocessing import utility_functions as uf
from eye_tracking_system_tools.preprocessing import load_aligned_arena_data
from eye_tracking_system_tools.preprocessing.arena_alignment import (
    get_arena_alignment_constants,
    get_arena_video_frame_offset,
)
from eye_tracking_system_tools.preprocessing.block_sync_core import load_final_sync_df

from matplotlib import rcParams
%matplotlib inline
plt.style.use('default')
rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42

In [9]:
# Configuration
animal = "PV_208"
date = "2025_12_14"
block_num = "019"

base_path = Path(r"D:\sample_data_for_eye_repo")
block_path = base_path / animal / date / f"block_{block_num}"
arena_videos_dir = block_path / "arena_videos"

# LFP channel to use
lfp_channel = 21

print(f"Block path: {block_path}")
print(f"Arena videos dir: {arena_videos_dir}")

Block path: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019
Arena videos dir: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019\arena_videos


In [10]:
# Load block
block = BlockSync(animal,date,block_num,base_path)
block.channeldict = {1: "LED_driver", 7: "L_eye_TTL", 2: "Arena_TTL", 8: "R_eye_TTL"}
block.parse_open_ephys_events()

# Load final sync df
load_final_sync_df(block)
block.final_sync_df['ms_axis'] = block.final_sync_df['Arena_TTL'].values / (block.sample_rate / 1000)

# Load eye data
block.left_eye_data = pd.read_csv(block.analysis_path / 'left_eye_data_degrees_raw_verified.csv')
block.right_eye_data = pd.read_csv(block.analysis_path / 'right_eye_data_degrees_raw_verified.csv')

# Handle videos and calibrate
block.handle_eye_videos()
block.handle_arena_files()
block.calibrate_pixel_size(10)

# Calculate pupil diameter if needed
if 'pupil_diameter' not in block.left_eye_data.columns:
    print('Calculating pupil diameter...')
    block.left_eye_data['pupil_diameter_pixels'] = block.left_eye_data.major_ax
    block.right_eye_data['pupil_diameter_pixels'] = block.right_eye_data.major_ax
    block.left_eye_data['pupil_diameter'] = block.left_eye_data['pupil_diameter_pixels'] * block.L_pix_size
    block.right_eye_data['pupil_diameter'] = block.right_eye_data['pupil_diameter_pixels'] * block.R_pix_size

print(f"Sample rate: {block.sample_rate} Hz")
print(f"OE events shape: {block.oe_events.shape}")

instantiated block number 019 at Path: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019, new OE version
Found the sample rate for block 019 in the xml file, it is 20000 Hz

Extracting meta data from: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019\oe_files\PV208_d5t2_2025-12-14_12-29-53\Record Node 106...

Extracting time stamp information...

Error!!! Some blocks are missing in recording!!!

Checking integrity of all records in ch1...

Metadata extraction complete.
created the .oe_rec attribute as an open ephys recording obj with get_data functionality (standalone mode)
retrieving zertoh sample number for block 019
got it!
running parse_open_ephys_events...
block 019 has a parsed events file, reading...
[INFO] Found multiple sync files: ['final_sync_df.csv', 'blocksync_df.csv']. Using newest: final_sync_df.csv
[OK] Loaded final_sync_df.csv → block.final_sync_df (rows=113,056)
handling eye video files
converting videos...
h264 files found: 2; already have .mp4 (skip): 2;

In [11]:
def recenter_eye_angles_to_rest(
        eye_df: pd.DataFrame,
        phi_col: str = "k_phi",
        theta_col: str = "k_theta",
        inplace: bool = False,
        dropna: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """Recenter eye angles so median becomes zero."""
    df = eye_df if inplace else eye_df.copy()
    phi_vals = pd.to_numeric(df[phi_col], errors="coerce")
    theta_vals = pd.to_numeric(df[theta_col], errors="coerce")
    phi_med = float(np.nanmedian(phi_vals)) if dropna else float(np.median(phi_vals))
    theta_med = float(np.nanmedian(theta_vals)) if dropna else float(np.median(theta_vals))
    df[phi_col] = phi_vals - phi_med
    df[theta_col] = theta_vals - theta_med
    df[f"{phi_col}_recentered"] = df[phi_col]
    df[f"{theta_col}_recentered"] = df[theta_col]
    offsets = {"phi_offset": phi_med, "theta_offset": theta_med}
    return df, offsets

# Apply recentering
df_left_centered, left_offsets = recenter_eye_angles_to_rest(block.left_eye_data.copy())
df_right_centered, right_offsets = recenter_eye_angles_to_rest(block.right_eye_data.copy())
block.left_eye_data_centered = df_left_centered
block.right_eye_data_centered = df_right_centered

print("Left eye offsets:", left_offsets)
print("Right eye offsets:", right_offsets)

Left eye offsets: {'phi_offset': 5.636328644563306, 'theta_offset': -40.0718383983085}
Right eye offsets: {'phi_offset': -0.8638616368366938, 'theta_offset': -36.10163651109649}


### Arena alignment diagnostic

Two-clock synchronization: Arena PC clock (all CSV files) vs Open Ephys clock (video frames).
The alignment matches the first arena video frame between the two clocks. Run the cell below
to see the alignment constants and verify camera timestamp accuracy (should be < 100 ms difference).

In [12]:
# Arena alignment constants (two-clock synchronization)
constants = get_arena_alignment_constants(block, arena_videos_dir)
arena_frame_offset = get_arena_video_frame_offset(block, arena_videos_dir)
sync_ms = block.final_sync_df["ms_axis"]

print("Two-clock synchronization:")
print(f"  OE clock: first arena frame = {constants['first_arena_frame_oe_ms']:.2f} ms (from recording start)")
print(f"  PC clock: first arena frame = {constants['first_arena_frame_pc_ms']:.2f} ms (median)")
print(f"  Camera accuracy: max difference = {constants['camera_accuracy_ms']:.2f} ms")
print(f"  Shift: {constants['shift_ms']:.2f} ms (OE_ms = PC_ms - shift_ms)")
print(f"\nAll camera first-frame times (ms):")
for i, t_ms in enumerate(constants['all_first_frame_pc_ms']):
    print(f"    Camera {i+1}: {t_ms:.2f} ms")

print(f"\nfinal_sync_df ms_axis range: {sync_ms.min():.0f} - {sync_ms.max():.0f} ms")
print(f"  (video frames are indexed by this range)")
print(f"\nArena_frame → video file frame offset: {arena_frame_offset} (video_frame_index = Arena_frame + offset)")

Two-clock synchronization:
  OE clock: first arena frame = 15.40 ms (from recording start)
  PC clock: first arena frame = 1765708230354.76 ms (median)
  Camera accuracy: max difference = 7.73 ms
  Shift: 1765708230339.36 ms (OE_ms = PC_ms - shift_ms)

All camera first-frame times (ms):
    Camera 1: 1765708230358.95 ms
    Camera 2: 1765708230354.25 ms
    Camera 3: 1765708230355.28 ms
    Camera 4: 1765708230351.22 ms

final_sync_df ms_axis range: 15 - 1882381 ms
  (video frames are indexed by this range)


## Data format reference

- **final_sync_df**: `Arena_TTL` = OE sample (0 = recording start); `Arena_frame` = frame index from sync scheme (0-based from first TTL). ms_axis = (Arena_TTL / sample_rate) * 1000.
- **frames_timestamps CSVs**: Column 0 = frame index (0 = first video frame); column 1 = Unix seconds (PC clock). Video files match these CSVs by name.
- **Intended mapping**: After alignment, the same frame index n in final_sync_df (Arena_frame) and in the video file should correspond to the same ms_axis. Use `video_frame_index = Arena_frame + arena_frame_offset` (offset computed above).

## Diagnostic: Verify trial times vs arena frame selection

Check that trial start/end times correctly map to arena video frames.

In [13]:
# Load aligned arena data (two-clock synchronization: PC clock → OE clock)
arena_data = load_aligned_arena_data(block, arena_videos_dir)

trials_df = arena_data.get('trials_data', None)
bug_traj_df = arena_data.get('bug_trajectory', None)

if trials_df is None:
    raise ValueError("trials_data.csv not found in arena_videos folder")
if bug_traj_df is None:
    raise ValueError("bug_trajectory.csv not found in arena_videos folder")

print(f"Loaded {len(trials_df)} trials")
print(f"Bug trajectory: {len(bug_traj_df)} data points")
print(f"\nTrials columns: {list(trials_df.columns)}")
print(f"Bug trajectory columns: {list(bug_traj_df.columns)}")

[Arena alignment] First frame PC-clock: 1765708230354.76 ms (median), cameras differ by max 7.73 ms
[Arena alignment] First frame OE-clock: 15.40 ms
[Arena alignment] Shift: 1765708230339.36 ms (OE_ms = PC_ms - shift_ms)
Loaded 15 trials
Bug trajectory: 65901 data points

Trials columns: ['Unnamed: 0', 'trial_db_id', 'start_time', 'trial_bugs', 'bug_sizes', 'bug_speed', 'exit_hole', 'extra', 'duration', 'end_time', 'ms_axis_start', 'ms_axis_end']
Bug trajectory columns: ['Unnamed: 0', 'time', 'x', 'y', 'ms_axis']


In [15]:
# Diagnostic: Check first trial's time mapping AND verify Arena_frame offset
if len(trials_df) > 0:
    first_trial = trials_df.iloc[0]
    trial_start_ms = float(first_trial['ms_axis_start'])
    trial_end_ms = float(first_trial['ms_axis_end'])
    
    print(f"First trial: {first_trial.get('trial_db_id', '?')}")
    print(f"  Trial start (ms_axis): {trial_start_ms:.2f} ms")
    print(f"  Trial end (ms_axis): {trial_end_ms:.2f} ms")
    print(f"  Duration: {trial_end_ms - trial_start_ms:.2f} ms ({ (trial_end_ms - trial_start_ms)/1000:.2f} s)")
    
    # Use single helper for Arena_frame → video file frame offset
    print(f"\n  Checking Arena_frame to video frame mapping...")
    arena_frame_offset = get_arena_video_frame_offset(block, arena_videos_dir)
    print(f"    Frame offset (get_arena_video_frame_offset): {arena_frame_offset} frames")
    
    # Load left arena timestamps for expected video frame range
    frames_ts_dir = arena_videos_dir / "videos" / "frames_timestamps"
    left_ts_files = list(frames_ts_dir.glob("*left*.csv"))
    if left_ts_files:
        left_ts_path = left_ts_files[0]
        left_ts_df = pd.read_csv(left_ts_path, header=0)
        unix_s_col = left_ts_df.columns[1]
        left_ts_df['pc_ms'] = left_ts_df[unix_s_col].values * 1000.0
        frame_times_pc_ms = left_ts_df['pc_ms'].values
        shift_ms = constants['shift_ms']
        
        # Check trial frames
        print(f"\n  Checking trial frame selection...")
        fsync = block.final_sync_df
        ms_all = fsync['ms_axis'].to_numpy(dtype=float)
        mask = np.isfinite(ms_all) & (ms_all >= trial_start_ms) & (ms_all <= trial_end_ms)
        trial_sync_rows = fsync[mask]
        
        if len(trial_sync_rows) > 0:
            # Coerce to numeric (invalid -> NaN), dropna, then keep only non-negative frame indices
            af = pd.to_numeric(trial_sync_rows['Arena_frame'], errors='coerce')
            arena_frames_valid = af.dropna()
            arena_frames_valid = arena_frames_valid[arena_frames_valid >= 0]
            if len(arena_frames_valid) > 0:
                arena_frames = arena_frames_valid.astype(int)
                print(f"    Arena_frame from final_sync_df: {arena_frames.min()} to {arena_frames.max()} (valid: {len(arena_frames)}/{len(trial_sync_rows)} rows)")
            else:
                print(f"    WARNING: No valid Arena_frame values in trial window!")
                arena_frames = pd.Series(dtype=int)
            
            # Convert trial times to PC-clock
            trial_start_pc_ms = trial_start_ms + shift_ms
            trial_end_pc_ms = trial_end_ms + shift_ms
            
            # Find expected video frames
            start_frame_idx = np.argmin(np.abs(frame_times_pc_ms - trial_start_pc_ms))
            end_frame_idx = np.argmin(np.abs(frame_times_pc_ms - trial_end_pc_ms))
            
            print(f"    Expected video frames (from frames_timestamps): {start_frame_idx} to {end_frame_idx}")
            
            if len(arena_frames) > 0:
                # Apply offset (same as get_arena_video_frame_offset)
                frame_offset = arena_frame_offset
                corrected_start = int(arena_frames.min()) + frame_offset
                corrected_end = int(arena_frames.max()) + frame_offset
                
                print(f"\n    After applying frame offset ({frame_offset}):")
                print(f"      Corrected Arena_frame: {corrected_start} to {corrected_end}")
                print(f"      Expected video frames: {start_frame_idx} to {end_frame_idx}")
                
                diff_start = abs(corrected_start - start_frame_idx)
                diff_end = abs(corrected_end - end_frame_idx)
            else:
                print(f"\n    Cannot compare: no valid Arena_frame values")
                diff_start = float('inf')
                diff_end = float('inf')
            print(f"      Difference: start={diff_start} frames, end={diff_end} frames")
            
            if diff_start > 2 or diff_end > 2:
                print(f"\n  ⚠️  WARNING: Frame mismatch even after offset correction!")
            else:
                print(f"\n  ✓ Frame mapping verified after offset correction")
        else:
            print("  ⚠️  No final_sync_df rows found in trial window!")

First trial: 10097
  Trial start (ms_axis): 10297.64 ms
  Trial end (ms_axis): 92156.64 ms
  Duration: 81859.00 ms (81.86 s)

  Checking Arena_frame to video frame mapping...
    First Arena_TTL sample: 308.0
    First Arena_TTL_frame: 0.0
    First Arena_TTL ms_axis: 15.40 ms

    First Arena_TTL maps to video frame: 0
    Arena_TTL_frame value: 0.0
    Frame offset: 0.0 frames

  Checking trial frame selection...
    Arena_frame from final_sync_df: 0 to 3279 (valid: 3335/4917 rows)
    Expected video frames (from frames_timestamps): 608 to 5444

    After applying frame offset (0):
      Corrected Arena_frame: 0 to 3279
      Expected video frames: 608 to 5444
      Difference: start=608 frames, end=2165 frames

  ⚠️  WARNING: Frame mismatch even after offset correction!


### Bug trajectory conversion check (same logic as arena_sync_verification.ipynb)

Verify that `bug_traj_df['ms_axis']` matches the conversion used in the verification notebook:
raw `time` → `arena_datetime_to_pc_ms` → OE_ms = PC_ms - shift_ms.

In [16]:
# Cross-check: reproduce verification-notebook conversion and compare to bug_traj_df
from eye_tracking_system_tools.preprocessing.arena_alignment import arena_datetime_to_pc_ms

constants = get_arena_alignment_constants(block, arena_videos_dir)
shift_ms = constants['shift_ms']

# Raw bug_trajectory (same as verification notebook)
bug_raw = pd.read_csv(arena_videos_dir / "bug_trajectory.csv")
bug_raw["time_pc_ms"] = arena_datetime_to_pc_ms(bug_raw["time"])
bug_raw["ms_axis_manual"] = bug_raw["time_pc_ms"] - shift_ms

# Compare first few rows: bug_traj_df vs manual conversion
n_check = min(5, len(bug_traj_df), len(bug_raw))
print("Bug trajectory conversion check (same logic as arena_sync_verification.ipynb):")
print(f"  shift_ms = {shift_ms:.2f}")
print(f"  Formula: ms_axis = arena_datetime_to_pc_ms(time) - shift_ms")
print(f"\n  First {n_check} rows comparison:")
print(f"  {'idx':<4} {'bug_traj_df ms_axis':<22} {'manual ms_axis':<22} {'diff (ms)'}")
for i in range(n_check):
    from_load = bug_traj_df["ms_axis"].iloc[i]
    manual = bug_raw["ms_axis_manual"].iloc[i]
    diff = from_load - manual
    print(f"  {i:<4} {from_load:<22.2f} {manual:<22.2f} {diff:.4f}")
if n_check > 0:
    max_diff = np.abs(bug_traj_df["ms_axis"].iloc[:n_check].values - bug_raw["ms_axis_manual"].iloc[:n_check].values).max()
    if max_diff > 0.01:
        print(f"\n  WARNING: max difference = {max_diff:.4f} ms — conversion may differ from verification!")
    else:
        print(f"\n  OK: bug_traj_df ms_axis matches verification conversion (diff < 0.01 ms)")

Bug trajectory conversion check (same logic as arena_sync_verification.ipynb):
  shift_ms = 1765708230339.36
  Formula: ms_axis = arena_datetime_to_pc_ms(time) - shift_ms

  First 5 rows comparison:
  idx  bug_traj_df ms_axis    manual ms_axis         diff (ms)
  0    10311.64               10311.64               0.0000
  1    10327.64               10327.64               0.0000
  2    10343.64               10343.64               0.0000
  3    10359.64               10359.64               0.0000
  4    10375.64               10375.64               0.0000

  OK: bug_traj_df ms_axis matches verification conversion (diff < 0.01 ms)


## Video Export Function

Modified version of `export_block_synchronized_montage_video` that includes bug trajectory and LFP traces.

In [17]:
class MonotoneFrameReader:
    """Frame-exact reader for mostly-nondecreasing frame indices."""
    def __init__(self, path, label="video"):
        self.path = str(path)
        self.label = label
        self.cap = cv2.VideoCapture(self.path)
        if not self.cap.isOpened():
            raise RuntimeError(f"Cannot open {label}: {path}")
        self.cur_idx = -1
        self.cur_frame = None

    def close(self):
        try:
            self.cap.release()
        except Exception:
            pass

    def _reopen_and_seek(self, target_idx: int):
        self.close()
        self.cap = cv2.VideoCapture(self.path)
        if not self.cap.isOpened():
            raise RuntimeError(f"Cannot reopen {self.label}: {self.path}")
        self.cur_idx = -1
        self.cur_frame = None
        if target_idx > 0:
            for _ in range(target_idx):
                ok = self.cap.grab()
                if not ok:
                    return None

    def read_at(self, target_idx: Optional[int]):
        if target_idx is None or target_idx < 0:
            return None
        target_idx = int(target_idx)
        if target_idx == self.cur_idx and self.cur_frame is not None:
            return self.cur_frame
        if target_idx < self.cur_idx:
            self._reopen_and_seek(target_idx)
        while self.cur_idx < target_idx:
            ok, frame = self.cap.read()
            if not ok:
                return None
            self.cur_idx += 1
            self.cur_frame = frame
        return self.cur_frame

In [18]:
import os

## Rolling-window trace view (caret centered, 5 s before/after)

Alternative export: trace panel shows a **rolling window** of 5 s before and 5 s after the current time; the caret stays fixed at center and the data scrolls. Use `export_trial_video_rolling_window` below.

In [19]:
def export_trial_video_rolling_window(
    block: object,
    trial_row: pd.Series,
    bug_traj_df: pd.DataFrame,
    out_path: Union[Path, str],
    *,
    fps: float = 60.0,
    lfp_channel: int = 1,
    trace_window_half_s: float = 5.0,
    arena_video: Optional[Union[int, str]] = None,
    top_banner_h: int = 60,
    trace_h: int = 300,
    trace_scale: float = 2.0,
    flip_eyes_vertical: bool = True,
    codec: str = "mp4v",
    show_debug_prints: bool = True,
) -> Path:
    """
    Export a single trial video with rolling-window trace panel.
    Caret is fixed at center; trace shows (trace_window_half_s) seconds before and after current time.
    """
    trace_window_half_ms = trace_window_half_s * 1000.0

    start_ms = float(trial_row['ms_axis_start'])
    end_ms = float(trial_row['ms_axis_end'])
    if end_ms <= start_ms:
        raise ValueError(f"Invalid trial time range: {start_ms} to {end_ms} ms")

    fs = block.sample_rate
    fsync = block.final_sync_df
    ms_all = fsync['ms_axis'].to_numpy(dtype=float)
    mask = np.isfinite(ms_all) & (ms_all >= start_ms) & (ms_all <= end_ms)
    if not np.any(mask):
        raise ValueError(f"No final_sync_df rows in trial window [{start_ms}, {end_ms}] ms")

    idx_rows = np.where(mask)[0]
    fs_win = fsync.iloc[idx_rows].copy()
    t_ms = ms_all[idx_rows].astype(float)
    t_ms_rel = t_ms - start_ms

    if len(t_ms) > 5:
        dt = np.median(np.diff(t_ms))
        fps_master = 1000.0 / dt if dt > 0 else float("nan")
        if np.isfinite(fps_master) and fps_master > 0:
            stride = int(round(fps_master / float(fps))) if float(fps) <= fps_master else 1
            stride = max(1, stride)
            if stride > 1:
                fs_win = fs_win.iloc[::stride].copy()
                t_ms = t_ms[::stride]
                t_ms_rel = t_ms_rel[::stride]

    # Filter out rows with invalid Arena_frame (NaN or negative) before extracting frame indices
    af_num = pd.to_numeric(fs_win['Arena_frame'], errors='coerce')
    valid_mask = (af_num.notna()) & (af_num >= 0)
    fs_win_valid = fs_win[valid_mask].copy()
    t_ms = t_ms[valid_mask.values]
    t_ms_rel = t_ms_rel[valid_mask.values]
    
    A_frames = fs_win_valid['Arena_frame'].astype(int).to_numpy()
    L_frames = fs_win_valid['L_eye_frame'].to_numpy(dtype=int)
    R_frames = fs_win_valid['R_eye_frame'].to_numpy(dtype=int)
    
    # Arena_frame in final_sync_df is 0-based from first TTL; video file frame 0 is first frame of file
    arena_frame_offset = get_arena_video_frame_offset(block, block.block_path / "arena_videos")
    A_frames = A_frames + arena_frame_offset

    left_df = block.left_eye_data_centered
    right_df = block.right_eye_data_centered
    Ltab = left_df.drop_duplicates(subset=["eye_frame"], keep="first").set_index("eye_frame", drop=False)
    Rtab = right_df.drop_duplicates(subset=["eye_frame"], keep="first").set_index("eye_frame", drop=False)
    L_eye_idx = pd.Index(L_frames, name="eye_frame")
    R_eye_idx = pd.Index(R_frames, name="eye_frame")

    L_phi = Ltab.reindex(L_eye_idx)['k_phi_recentered'].to_numpy(dtype=float)
    R_phi = Rtab.reindex(R_eye_idx)['k_phi_recentered'].to_numpy(dtype=float)
    L_theta = Ltab.reindex(L_eye_idx)['k_theta_recentered'].to_numpy(dtype=float)
    R_theta = Rtab.reindex(R_eye_idx)['k_theta_recentered'].to_numpy(dtype=float)
    L_pupil = Ltab.reindex(L_eye_idx)['pupil_diameter'].to_numpy(dtype=float)
    R_pupil = Rtab.reindex(R_eye_idx)['pupil_diameter'].to_numpy(dtype=float)

    bug_mask = (bug_traj_df['ms_axis'] >= start_ms) & (bug_traj_df['ms_axis'] <= end_ms)
    bug_trial = bug_traj_df.loc[bug_mask].copy()
    bug_trial = bug_trial.sort_values('ms_axis').reset_index(drop=True)
    bug_t_ms_rel = bug_trial['ms_axis'].values - start_ms
    if len(bug_trial) > 0:
        bug_x_interp = np.interp(t_ms_rel, bug_t_ms_rel, bug_trial['x'].values)
        bug_y_interp = np.interp(t_ms_rel, bug_t_ms_rel, bug_trial['y'].values)
    else:
        bug_x_interp = np.full_like(t_ms_rel, np.nan)
        bug_y_interp = np.full_like(t_ms_rel, np.nan)

    window_ms = end_ms - start_ms
    global_start_ms = float(getattr(block.oe_rec, 'globalStartTime_ms', 0))
    oe_start_ms = start_ms + global_start_ms
    start_arr = np.atleast_2d(np.array([oe_start_ms], dtype=float))
    try:
        lfp_data, lfp_timestamps = block.oe_rec.get_data(
            channels=[lfp_channel],
            start_time_ms=start_arr,
            window_ms=window_ms,
            convert_microvolts=True,
            return_timestamps=True,
            repress_output=True,
        )
        if lfp_data is None or lfp_data.size == 0:
            raise ValueError("No LFP data returned")
        lfp_trace = lfp_data[0, 0, :]
        lfp_t_ms = lfp_timestamps[0, :]
        lfp_t_ms_rel = lfp_t_ms - lfp_t_ms[0]
    except Exception as e:
        print(f"Warning: Could not load LFP data: {e}")
        lfp_trace = None
        lfp_t_ms_rel = None

    rv_raw = Path(block.re_videos[0])
    lv_raw = Path(block.le_videos[0])
    arena_path = Path(block.arena_videos[0] if arena_video is None else block.arena_videos[arena_video])
    capR = cv2.VideoCapture(str(rv_raw))
    capL = cv2.VideoCapture(str(lv_raw))
    capA = cv2.VideoCapture(str(arena_path))
    rR = MonotoneFrameReader(rv_raw, "right_eye")
    rL = MonotoneFrameReader(lv_raw, "left_eye")
    rA = MonotoneFrameReader(arena_path, "arena")

    Wr, Hr = int(capR.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capR.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Wl, Hl = int(capL.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capL.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Wa, Ha = int(capA.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capA.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Heye = int(min(Hr, Hl))
    trace_h_eff = max(1, int(round(float(trace_h) * float(trace_scale))))
    Wr_out = max(1, int(round(Wr * (Heye / float(Hr)))))
    Wl_out = max(1, int(round(Wl * (Heye / float(Hl)))))
    Wa_out = max(1, int(round(Wa * (Heye / float(Ha)))))
    Hrow = Heye
    Wtotal = Wr_out + Wa_out + Wl_out
    Htotal = top_banner_h + Hrow + trace_h_eff

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*codec), float(fps), (Wtotal, Htotal))
    if not writer.isOpened():
        raise RuntimeError(f"Could not open VideoWriter for: {out_path}")

    def _safe_put_text(img, text, org, color, scale=0.6, thickness=2):
        x, y = org
        cv2.putText(img, text, (x + 1, y + 1), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), thickness + 2, cv2.LINE_AA)
        cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)

    def _make_banner(W: int, title: str) -> np.ndarray:
        banner = np.zeros((top_banner_h, W, 3), dtype=np.uint8)
        (tw, _), _ = cv2.getTextSize(title, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
        x = max(12, (W - tw) // 2)
        _safe_put_text(banner, title, (x, 34), (255, 255, 255), scale=0.9, thickness=2)
        return banner

    def _resize_to_height(img: np.ndarray, target_h: int) -> np.ndarray:
        h, w = img.shape[:2]
        if h == target_h:
            return img
        new_w = max(1, int(round(w * (target_h / float(h)))))
        return cv2.resize(img, (new_w, target_h), interpolation=cv2.INTER_AREA)

    def _map_to_axis(vals: np.ndarray, lo: float, hi: float) -> np.ndarray:
        return (vals - lo) / (hi - lo + 1e-12)

    def _limits_minmax_std(x: np.ndarray) -> Tuple[float, float]:
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        if x.size < 2:
            return (-1.0, 1.0)
        mn, mx = float(np.min(x)), float(np.max(x))
        sd = float(np.std(x))
        if not np.isfinite(sd) or sd < 1e-12:
            sd = max(1.0, 0.05 * (mx - mn) if (mx - mn) > 0 else 1.0)
        lo, hi = mn - sd, mx + sd
        if not np.isfinite(lo) or not np.isfinite(hi) or abs(hi - lo) < 1e-12:
            lo, hi = mn - 1.0, mx + 1.0
        if abs(hi - lo) < 1e-12:
            lo -= 1.0
            hi += 1.0
        return lo, hi

    def _draw_trace_panel_rolling(W: int, t_ms: float, t_grid: np.ndarray,
                                  L_phi: np.ndarray, R_phi: np.ndarray,
                                  L_theta: np.ndarray, R_theta: np.ndarray,
                                  L_pupil: np.ndarray, R_pupil: np.ndarray,
                                  bug_x: np.ndarray, bug_y: np.ndarray,
                                  lfp_trace: Optional[np.ndarray], lfp_t_ms: Optional[np.ndarray],
                                  trace_window_half_ms: float, trace_h_eff: int) -> np.ndarray:
        """Draw trace panel with rolling window: caret at center, window = t_ms ± trace_window_half_ms."""
        panel = np.zeros((trace_h_eff, W, 3), dtype=np.uint8)
        w0 = t_ms - trace_window_half_ms
        w1 = t_ms + trace_window_half_ms
        idx = np.where((t_grid >= w0) & (t_grid <= w1))[0]
        if idx.size < 2:
            cursor_x = W // 2
            cv2.line(panel, (cursor_x, 0), (cursor_x, trace_h_eff - 1), (120, 120, 120), 1)
            _safe_put_text(panel, f"t = {t_ms:.0f} ms", (12, 26), (255, 255, 255), scale=0.7, thickness=2)
            return panel

        tg = t_grid[idx]
        x = (tg - w0) / (w1 - w0 + 1e-12)
        xpix = (x * (W - 1)).astype(int)

        n_axes = 9 if lfp_trace is not None else 8
        pad_y = 12
        axis_h = max(50, (trace_h_eff - 2 * pad_y) // max(1, n_axes))
        color_L, color_R = (255, 0, 0), (0, 0, 255)
        color_bug, color_lfp = (0, 255, 0), (255, 255, 0)

        traces = [
            ("phi_L", L_phi[idx], color_L, "k_phi L"),
            ("phi_R", R_phi[idx], color_R, "k_phi R"),
            ("theta_L", L_theta[idx], color_L, "k_theta L"),
            ("theta_R", R_theta[idx], color_R, "k_theta R"),
            ("pupil_L", L_pupil[idx], color_L, "pupil L"),
            ("pupil_R", R_pupil[idx], color_R, "pupil R"),
            ("bug_x", bug_x[idx], color_bug, "bug_x"),
            ("bug_y", bug_y[idx], color_bug, "bug_y"),
        ]
        if lfp_trace is not None and lfp_t_ms is not None:
            lfp_interp = np.interp(tg, lfp_t_ms, lfp_trace)
            traces.append(("lfp", lfp_interp, color_lfp, "LFP"))

        for j, (name, vals, color, label) in enumerate(traces):
            y0 = pad_y + j * axis_h
            y1 = min(trace_h_eff - pad_y, y0 + axis_h) - 10
            yy0, yy1 = int(y0 + 20), int(y1 - 10)
            Hax = max(2, yy1 - yy0)
            cv2.rectangle(panel, (0, y0), (W - 1, y1), (20, 20, 20), 1)
            vals_f = vals.astype(float)
            lo, hi = _limits_minmax_std(vals_f)
            vals_n = _map_to_axis(vals_f, lo, hi)
            y_f = yy0 + (1.0 - np.clip(vals_n, 0.0, 1.0)) * (Hax - 1)
            y_i = y_f.astype(np.int32)
            m = np.isfinite(y_f)
            for k in range(1, len(xpix)):
                if m[k - 1] and m[k]:
                    cv2.line(panel, (int(xpix[k - 1]), int(y_i[k - 1])), (int(xpix[k]), int(y_i[k])), color, 1, cv2.LINE_AA)
            _safe_put_text(panel, f"{label} [{lo:.2f},{hi:.2f}]", (12, y0 + 18), (200, 200, 200), scale=0.5, thickness=1)

        cursor_x = W // 2
        cv2.line(panel, (cursor_x, 0), (cursor_x, trace_h_eff - 1), (120, 120, 120), 1)
        _safe_put_text(panel, f"t = {t_ms:.0f} ms  [±{trace_window_half_ms/1000:.1f}s]", (12, 26), (255, 255, 255), scale=0.7, thickness=2)
        return panel

    banner = _make_banner(Wtotal, f"Trial {trial_row.get('trial_db_id', '?')} (rolling ±{trace_window_half_s}s)")
    prev_R = np.zeros((Hr, Wr, 3), dtype=np.uint8)
    prev_L = np.zeros((Hl, Wl, 3), dtype=np.uint8)
    prev_A = np.zeros((Ha, Wa, 3), dtype=np.uint8)

    try:
        for i in tqdm(range(len(t_ms)), desc="Exporting trial video (rolling)", unit="frame"):
            tcur_rel = float(t_ms_rel[i])
            idxR = int(R_frames[i]) if R_frames[i] >= 0 else None
            idxL = int(L_frames[i]) if L_frames[i] >= 0 else None
            idxA = int(A_frames[i]) if A_frames[i] >= 0 else None
            fR = rR.read_at(idxR) if idxR is not None else prev_R.copy()
            fL = rL.read_at(idxL) if idxL is not None else prev_L.copy()
            fA = rA.read_at(idxA) if idxA is not None else prev_A.copy()
            if fR is not None:
                prev_R = fR.copy()
            if fL is not None:
                prev_L = fL.copy()
            if fA is not None:
                prev_A = fA.copy()
            if flip_eyes_vertical:
                fR = cv2.flip(fR, 0)
                fL = cv2.flip(fL, 0)
            _safe_put_text(fR, "RIGHT", (12, 24), (255, 255, 255), scale=0.75, thickness=2)
            _safe_put_text(fA, "ARENA", (12, 24), (255, 255, 255), scale=0.75, thickness=2)
            _safe_put_text(fL, "LEFT", (12, 24), (255, 255, 255), scale=0.75, thickness=2)
            fR = _resize_to_height(fR, Heye)
            fA = _resize_to_height(fA, Heye)
            fL = _resize_to_height(fL, Heye)
            row_img = np.concatenate([fR, fA, fL], axis=1)
            if row_img.shape[1] != Wtotal:
                if row_img.shape[1] < Wtotal:
                    row_img = cv2.copyMakeBorder(row_img, 0, 0, 0, Wtotal - row_img.shape[1], cv2.BORDER_CONSTANT, value=(0, 0, 0))
                else:
                    row_img = row_img[:, :Wtotal, :]

            trace = _draw_trace_panel_rolling(
                W=Wtotal, t_ms=tcur_rel, t_grid=t_ms_rel,
                L_phi=L_phi, R_phi=R_phi, L_theta=L_theta, R_theta=R_theta,
                L_pupil=L_pupil, R_pupil=R_pupil,
                bug_x=bug_x_interp, bug_y=bug_y_interp,
                lfp_trace=lfp_trace, lfp_t_ms=lfp_t_ms_rel,
                trace_window_half_ms=trace_window_half_ms,
                trace_h_eff=trace_h_eff,
            )

            frame = np.zeros((Htotal, Wtotal, 3), dtype=np.uint8)
            frame[0:top_banner_h, :, :] = banner
            frame[top_banner_h:top_banner_h + Hrow, :, :] = row_img
            frame[top_banner_h + Hrow:top_banner_h + Hrow + trace_h_eff, :, :] = trace
            writer.write(frame)
        return out_path
    finally:
        rR.close()
        rL.close()
        rA.close()
        try:
            writer.release()
        except Exception:
            pass
        for cap in (capR, capL, capA):
            try:
                cap.release()
            except Exception:
                pass

In [20]:
# Export trial videos with rolling-window trace view (caret centered, ±5 s)
output_dir_rolling = block.analysis_path / "trial_videos_rolling"
output_dir_rolling.mkdir(exist_ok=True)

# Optional: export only first N trials for testing (set to None to export all)
max_trials_rolling = None  # e.g. 2

for idx, trial_row in trials_df.iterrows():
    if max_trials_rolling is not None and idx >= max_trials_rolling:
        break
    trial_id = trial_row.get('trial_db_id', idx)
    bug_type = trial_row.get('bug_type', 'unknown')
    if pd.isna(trial_row.get('ms_axis_start')) or pd.isna(trial_row.get('ms_axis_end')):
        print(f"Skipping trial {trial_id}: missing time range")
        continue
    out_path = output_dir_rolling / f"trial_{trial_id:05d}_{bug_type}_rolling.mp4"
    try:
        print(f"\nExporting trial {trial_id} ({bug_type}) [rolling ±5s]...")
        export_trial_video_rolling_window(
            block=block,
            trial_row=trial_row,
            bug_traj_df=bug_traj_df,
            out_path=out_path,
            fps=60.0,
            lfp_channel=lfp_channel,
            trace_window_half_s=5.0,
            show_debug_prints=True,
            arena_video=2
        )
        print(f"  Saved: {out_path}")
    except Exception as e:
        print(f"  Failed: {e}")
        import traceback
        traceback.print_exc()

print(f"\nDone! Rolling-window videos saved to: {output_dir_rolling}")


Exporting trial 10097 (unknown) [rolling ±5s]...


c:\Users\nimro\miniconda3\envs\eye_repo\lib\site-packages\pandas\core\base.py:666: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)
Exporting trial video (rolling):   0%|          | 0/3335 [00:00<?, ?frame/s]C:\Users\nimro\AppData\Local\Temp\ipykernel_35568\4164682936.py:261: RuntimeWarning: invalid value encountered in cast
  y_i = y_f.astype(np.int32)
Exporting trial video (rolling): 100%|██████████| 3335/3335 [02:08<00:00, 26.03frame/s]
c:\Users\nimro\miniconda3\envs\eye_repo\lib\site-packages\pandas\core\base.py:666: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)


  Saved: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019\analysis\trial_videos_rolling\trial_10097_unknown_rolling.mp4

Exporting trial 10098 (unknown) [rolling ±5s]...


Exporting trial video (rolling):   0%|          | 0/1009 [00:53<?, ?frame/s]


KeyboardInterrupt: 

In [ ]:
def export_trial_video(
    block: object,
    trial_row: pd.Series,
    bug_traj_df: pd.DataFrame,
    out_path: Union[os.PathLike, str],
    *,
    fps: float = 60.0,
    lfp_channel: int = 1,
    arena_video: Optional[Union[int, str]] = None,
    top_banner_h: int = 60,
    trace_h: int = 300,  # Increased for more traces
    trace_scale: float = 2.0,
    flip_eyes_vertical: bool = True,
    codec: str = "mp4v",
    show_debug_prints: bool = True,
) -> Path:
    """
    Export a single trial video with bug trajectory and LFP traces.
    
    Parameters
    ----------
    block : BlockSync
        BlockSync instance with loaded data
    trial_row : pd.Series
        Single row from trials_data.csv with ms_axis_start and ms_axis_end
    bug_traj_df : pd.DataFrame
        Bug trajectory dataframe with ms_axis, x, y columns
    out_path : path-like
        Output video path
    fps : float
        Output video FPS
    lfp_channel : int
        LFP channel number to extract
    """
    import os
    
    start_ms = float(trial_row['ms_axis_start'])
    end_ms = float(trial_row['ms_axis_end'])
    
    if end_ms <= start_ms:
        raise ValueError(f"Invalid trial time range: {start_ms} to {end_ms} ms")
    
    fs = block.sample_rate
    fsync = block.final_sync_df
    
    # Filter final_sync_df to trial window
    ms_all = fsync['ms_axis'].to_numpy(dtype=float)
    mask = np.isfinite(ms_all) & (ms_all >= start_ms) & (ms_all <= end_ms)
    if not np.any(mask):
        raise ValueError(f"No final_sync_df rows in trial window [{start_ms}, {end_ms}] ms")
    
    idx_rows = np.where(mask)[0]
    fs_win = fsync.iloc[idx_rows].copy()
    t_ms = ms_all[idx_rows].astype(float)
    
    # Subsample to fps
    if len(t_ms) > 5:
        dt = np.median(np.diff(t_ms))
        fps_master = 1000.0 / dt if dt > 0 else float("nan")
        if np.isfinite(fps_master) and fps_master > 0:
            stride = int(round(fps_master / float(fps))) if float(fps) <= fps_master else 1
            stride = max(1, stride)
            if stride > 1:
                fs_win = fs_win.iloc[::stride].copy()
                t_ms = t_ms[::stride]
    
    # Filter out rows with invalid Arena_frame (NaN or negative) before extracting frame indices
    af_num = pd.to_numeric(fs_win['Arena_frame'], errors='coerce')
    valid_mask = (af_num.notna()) & (af_num >= 0)
    fs_win_valid = fs_win[valid_mask].copy()
    t_ms = t_ms[valid_mask.values]
    
    # Get frame indices; Arena_frame in final_sync_df is 0-based from first TTL
    A_frames = fs_win_valid['Arena_frame'].astype(int).to_numpy()
    L_frames = fs_win_valid['L_eye_frame'].to_numpy(dtype=int)
    R_frames = fs_win_valid['R_eye_frame'].to_numpy(dtype=int)
    
    arena_frame_offset = get_arena_video_frame_offset(block, block.block_path / "arena_videos")
    A_frames = A_frames + arena_frame_offset
    
    # Load eye trace data
    left_df = block.left_eye_data_centered
    right_df = block.right_eye_data_centered
    
    Ltab = left_df.drop_duplicates(subset=["eye_frame"], keep="first").set_index("eye_frame", drop=False)
    Rtab = right_df.drop_duplicates(subset=["eye_frame"], keep="first").set_index("eye_frame", drop=False)
    
    L_eye_idx = pd.Index(L_frames, name="eye_frame")
    R_eye_idx = pd.Index(R_frames, name="eye_frame")
    
    # Extract eye traces
    L_phi = Ltab.reindex(L_eye_idx)['k_phi_recentered'].to_numpy(dtype=float)
    R_phi = Rtab.reindex(R_eye_idx)['k_phi_recentered'].to_numpy(dtype=float)
    L_theta = Ltab.reindex(L_eye_idx)['k_theta_recentered'].to_numpy(dtype=float)
    R_theta = Rtab.reindex(R_eye_idx)['k_theta_recentered'].to_numpy(dtype=float)
    L_pupil = Ltab.reindex(L_eye_idx)['pupil_diameter'].to_numpy(dtype=float)
    R_pupil = Rtab.reindex(R_eye_idx)['pupil_diameter'].to_numpy(dtype=float)
    
    # Extract bug trajectory for this trial
    bug_mask = (bug_traj_df['ms_axis'] >= start_ms) & (bug_traj_df['ms_axis'] <= end_ms)
    bug_trial = bug_traj_df.loc[bug_mask].copy()
    bug_trial = bug_trial.sort_values('ms_axis').reset_index(drop=True)
    
    # Interpolate bug trajectory to match video frame times
    bug_x_interp = np.interp(t_ms, bug_trial['ms_axis'].values, bug_trial['x'].values)
    bug_y_interp = np.interp(t_ms, bug_trial['ms_axis'].values, bug_trial['y'].values)
    
    # Load LFP data (full resolution)
    window_ms = end_ms - start_ms
    global_start_ms = float(getattr(block.oe_rec, 'globalStartTime_ms', 0))
    
    # Convert ms_axis to OE timebase
    # ms_axis is ms from recording start, so add globalStartTime_ms
    oe_start_ms = start_ms + global_start_ms
    start_arr = np.atleast_2d(np.array([oe_start_ms], dtype=float))
    
    try:
        lfp_data, lfp_timestamps = block.oe_rec.get_data(
            channels=[lfp_channel],
            start_time_ms=start_arr,
            window_ms=window_ms,
            convert_microvolts=True,
            return_timestamps=True,
            repress_output=True,
        )
        if lfp_data is None or lfp_data.size == 0:
            raise ValueError("No LFP data returned")
        lfp_trace = lfp_data[0, 0, :]  # [n_channels, n_windows, n_samples]
        lfp_t_ms = lfp_timestamps[0, :]  # Timestamps in ms
        # Convert to relative ms from trial start
        lfp_t_ms_rel = lfp_t_ms - lfp_t_ms[0]
    except Exception as e:
        print(f"Warning: Could not load LFP data: {e}")
        lfp_trace = None
        lfp_t_ms_rel = None
    
    # Open videos
    rv_raw = Path(block.re_videos[0])
    lv_raw = Path(block.le_videos[0])
    arena_path = Path(block.arena_videos[0] if arena_video is None else block.arena_videos[arena_video])
    
    capR = cv2.VideoCapture(str(rv_raw))
    capL = cv2.VideoCapture(str(lv_raw))
    capA = cv2.VideoCapture(str(arena_path))
    
    rR = MonotoneFrameReader(rv_raw, "right_eye")
    rL = MonotoneFrameReader(lv_raw, "left_eye")
    rA = MonotoneFrameReader(arena_path, "arena")
    
    Wr, Hr = int(capR.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capR.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Wl, Hl = int(capL.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capL.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Wa, Ha = int(capA.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capA.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Output geometry
    Heye = int(min(Hr, Hl))
    trace_h_eff = max(1, int(round(float(trace_h) * float(trace_scale))))
    
    Wr_out = max(1, int(round(Wr * (Heye / float(Hr)))))
    Wl_out = max(1, int(round(Wl * (Heye / float(Hl)))))
    Wa_out = max(1, int(round(Wa * (Heye / float(Ha)))))
    
    Hrow = Heye
    Wtotal = Wr_out + Wa_out + Wl_out
    Htotal = top_banner_h + Hrow + trace_h_eff
    
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    
    writer = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*codec),
        float(fps),
        (Wtotal, Htotal),
    )
    
    if not writer.isOpened():
        raise RuntimeError(f"Could not open VideoWriter for: {out_path}")
    
    # Helper functions
    def _safe_put_text(img, text, org, color, scale=0.6, thickness=2):
        x, y = org
        cv2.putText(img, text, (x + 1, y + 1), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0),
                    thickness + 2, cv2.LINE_AA)
        cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)
    
    def _make_banner(W: int, title: str) -> np.ndarray:
        banner = np.zeros((top_banner_h, W, 3), dtype=np.uint8)
        (tw, _), _ = cv2.getTextSize(title, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
        x = max(12, (W - tw) // 2)
        _safe_put_text(banner, title, (x, 34), (255, 255, 255), scale=0.9, thickness=2)
        return banner
    
    def _resize_to_height(img: np.ndarray, target_h: int) -> np.ndarray:
        h, w = img.shape[:2]
        if h == target_h:
            return img
        new_w = max(1, int(round(w * (target_h / float(h)))))
        return cv2.resize(img, (new_w, target_h), interpolation=cv2.INTER_AREA)
    
    def _map_to_axis(vals: np.ndarray, lo: float, hi: float) -> np.ndarray:
        return (vals - lo) / (hi - lo + 1e-12)
    
    def _limits_minmax_std(x: np.ndarray) -> Tuple[float, float]:
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        if x.size < 2:
            return (-1.0, 1.0)
        mn, mx = float(np.min(x)), float(np.max(x))
        sd = float(np.std(x))
        if not np.isfinite(sd) or sd < 1e-12:
            sd = max(1.0, 0.05 * (mx - mn) if (mx - mn) > 0 else 1.0)
        lo, hi = mn - sd, mx + sd
        if not np.isfinite(lo) or not np.isfinite(hi) or abs(hi - lo) < 1e-12:
            lo, hi = mn - 1.0, mx + 1.0
        if abs(hi - lo) < 1e-12:
            lo -= 1.0
            hi += 1.0
        return lo, hi
    
    def _draw_trace_panel(W: int, t_ms: float, t_grid: np.ndarray,
                          L_phi: np.ndarray, R_phi: np.ndarray,
                          L_theta: np.ndarray, R_theta: np.ndarray,
                          L_pupil: np.ndarray, R_pupil: np.ndarray,
                          bug_x: np.ndarray, bug_y: np.ndarray,
                          lfp_trace: Optional[np.ndarray], lfp_t_ms: Optional[np.ndarray],
                          t0: float, t1: float, trace_h_eff: int) -> np.ndarray:
        """Draw trace panel with all signals."""
        panel = np.zeros((trace_h_eff, W, 3), dtype=np.uint8)
        
        w0, w1 = t0, t1
        idx = np.where((t_grid >= w0) & (t_grid <= w1))[0]
        if idx.size < 2:
            return panel
        
        tg = t_grid[idx]
        x = (tg - w0) / (w1 - w0 + 1e-12)
        xpix = (x * (W - 1)).astype(int)
        
        # 7 traces: phi_L, phi_R, theta_L, theta_R, pupil_L, pupil_R, bug_x, bug_y, LFP
        n_axes = 9 if lfp_trace is not None else 8
        pad_y = 12
        axis_h = max(50, (trace_h_eff - 2 * pad_y) // max(1, n_axes))
        
        color_L = (255, 0, 0)   # blue
        color_R = (0, 0, 255)   # red
        color_bug = (0, 255, 0)  # green
        color_lfp = (255, 255, 0)  # cyan
        
        traces = [
            ("phi_L", L_phi[idx], color_L, "k_phi L"),
            ("phi_R", R_phi[idx], color_R, "k_phi R"),
            ("theta_L", L_theta[idx], color_L, "k_theta L"),
            ("theta_R", R_theta[idx], color_R, "k_theta R"),
            ("pupil_L", L_pupil[idx], color_L, "pupil L"),
            ("pupil_R", R_pupil[idx], color_R, "pupil R"),
            ("bug_x", bug_x[idx], color_bug, "bug_x"),
            ("bug_y", bug_y[idx], color_bug, "bug_y"),
        ]
        
        if lfp_trace is not None and lfp_t_ms is not None:
            # Interpolate LFP to video frame times
            lfp_interp = np.interp(tg, lfp_t_ms, lfp_trace)
            traces.append(("lfp", lfp_interp, color_lfp, "LFP"))
        
        for j, (name, vals, color, label) in enumerate(traces):
            y0 = pad_y + j * axis_h
            y1 = min(trace_h_eff - pad_y, y0 + axis_h) - 10
            yy0, yy1 = int(y0 + 20), int(y1 - 10)
            Hax = max(2, yy1 - yy0)
            
            cv2.rectangle(panel, (0, y0), (W - 1, y1), (20, 20, 20), 1)
            
            vals_f = vals.astype(float)
            lo, hi = _limits_minmax_std(vals_f)
            vals_n = _map_to_axis(vals_f, lo, hi)
            
            y_f = yy0 + (1.0 - np.clip(vals_n, 0.0, 1.0)) * (Hax - 1)
            y_i = y_f.astype(np.int32)
            m = np.isfinite(y_f)
            
            for k in range(1, len(xpix)):
                if m[k - 1] and m[k]:
                    cv2.line(panel,
                            (int(xpix[k - 1]), int(y_i[k - 1])),
                            (int(xpix[k]), int(y_i[k])),
                            color, 1, cv2.LINE_AA)
            
            _safe_put_text(panel, f"{label} [{lo:.2f},{hi:.2f}]", (12, y0 + 18),
                         (200, 200, 200), scale=0.5, thickness=1)
        
        cursor_x = int(round((np.clip(t_ms, w0, w1) - w0) / (w1 - w0 + 1e-12) * (W - 1)))
        cv2.line(panel, (cursor_x, 0), (cursor_x, trace_h_eff - 1), (120, 120, 120), 1)
        _safe_put_text(panel, f"t = {t_ms:.0f} ms", (12, 26),
                     (255, 255, 255), scale=0.7, thickness=2)
        
        return panel
    
    banner = _make_banner(Wtotal, f"Trial {trial_row.get('trial_db_id', '?')} - {trial_row.get('bug_type', 'unknown')}")
    
    prev_R = np.zeros((Hr, Wr, 3), dtype=np.uint8)
    prev_L = np.zeros((Hl, Wl, 3), dtype=np.uint8)
    prev_A = np.zeros((Ha, Wa, 3), dtype=np.uint8)
    
    try:
        for i in tqdm(range(len(t_ms)), desc="Exporting trial video", unit="frame"):
            tcur = float(t_ms[i])
            
            idxR = int(R_frames[i]) if R_frames[i] >= 0 else None
            idxL = int(L_frames[i]) if L_frames[i] >= 0 else None
            idxA = int(A_frames[i]) if A_frames[i] >= 0 else None
            
            fR = rR.read_at(idxR) if idxR is not None else prev_R.copy()
            fL = rL.read_at(idxL) if idxL is not None else prev_L.copy()
            fA = rA.read_at(idxA) if idxA is not None else prev_A.copy()
            
            if fR is not None:
                prev_R = fR.copy()
            if fL is not None:
                prev_L = fL.copy()
            if fA is not None:
                prev_A = fA.copy()
            
            if flip_eyes_vertical:
                fR = cv2.flip(fR, 0)
                fL = cv2.flip(fL, 0)
            
            _safe_put_text(fR, "RIGHT", (12, 24), (255, 255, 255), scale=0.75, thickness=2)
            _safe_put_text(fA, "ARENA", (12, 24), (255, 255, 255), scale=0.75, thickness=2)
            _safe_put_text(fL, "LEFT", (12, 24), (255, 255, 255), scale=0.75, thickness=2)
            
            fR = _resize_to_height(fR, Heye)
            fA = _resize_to_height(fA, Heye)
            fL = _resize_to_height(fL, Heye)
            
            row_img = np.concatenate([fR, fA, fL], axis=1)
            if row_img.shape[1] != Wtotal:
                if row_img.shape[1] < Wtotal:
                    row_img = cv2.copyMakeBorder(row_img, 0, 0, 0, Wtotal - row_img.shape[1],
                                               cv2.BORDER_CONSTANT, value=(0, 0, 0))
                else:
                    row_img = row_img[:, :Wtotal, :]
            
            trace = _draw_trace_panel(
                W=Wtotal, t_ms=tcur, t_grid=t_ms,
                L_phi=L_phi, R_phi=R_phi,
                L_theta=L_theta, R_theta=R_theta,
                L_pupil=L_pupil, R_pupil=R_pupil,
                bug_x=bug_x_interp, bug_y=bug_y_interp,
                lfp_trace=lfp_trace, lfp_t_ms=lfp_t_ms_rel,
                t0=float(t_ms[0]), t1=float(t_ms[-1]),
                trace_h_eff=trace_h_eff,
            )
            
            frame = np.zeros((Htotal, Wtotal, 3), dtype=np.uint8)
            frame[0:top_banner_h, :, :] = banner
            frame[top_banner_h:top_banner_h + Hrow, :, :] = row_img
            frame[top_banner_h + Hrow:top_banner_h + Hrow + trace_h_eff, :, :] = trace
            
            writer.write(frame)
        
        return out_path
    
    finally:
        rR.close(); rL.close(); rA.close()
        try:
            writer.release()
        except Exception:
            pass
        for cap in (capR, capL, capA):
            try:
                cap.release()
            except Exception:
                pass

In [ ]:
block.arena_videos

In [ ]:
# Export videos for all trials
output_dir = block.analysis_path / "trial_videos"
output_dir.mkdir(exist_ok=True)

for idx, trial_row in trials_df.iterrows():
    trial_id = trial_row.get('trial_db_id', idx)
    bug_type = trial_row.get('bug_type', 'unknown')
    
    if pd.isna(trial_row.get('ms_axis_start')) or pd.isna(trial_row.get('ms_axis_end')):
        print(f"Skipping trial {trial_id}: missing time range")
        continue
    
    out_path = output_dir / f"trial_{trial_id:05d}_{bug_type}.mp4"
    
    try:
        print(f"\nExporting trial {trial_id} ({bug_type})...")
        export_trial_video(
            block=block,
            trial_row=trial_row,
            bug_traj_df=bug_traj_df,
            out_path=out_path,
            fps=60.0,
            lfp_channel=lfp_channel,
            show_debug_prints=True,
        )
        print(f"  Saved: {out_path}")
    except Exception as e:
        print(f"  Failed: {e}")
        import traceback
        traceback.print_exc()

print(f"\nDone! Videos saved to: {output_dir}")